In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 5.5
fig_height = 3.5
fig_format = 'pdf'
fig_dpi = 300
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"
  from IPython.display import set_matplotlib_formats
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'QzpcVXNlcnNcSmVyc29uXERvd25sb2Fkc1xsb2dpY2FpYQ=='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"c:\\Users\\Jerson\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\importlib\\_bootstrap.py": 1738705056.0, "c:\\Users\\Jerson\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\importlib\\_bootstrap_external.py": 1738705056.0, "c:\\Users\\Jerson\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\zipimport.py": 1738705056.0, "c:\\Users\\Jerson\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\codecs.py": 1738705056.0, "c:\\Users\\Jerson\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\encodings\\aliases.py": 1738705056.0, "c:\\Users\\Jerson\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\encodings\\__init__.py": 1738705056.0, "c:\\Users\\Jerson\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\encodings\\utf_8.py": 1738705056.0, "c:\\Users\\Jerson\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\encodings\\cp1252.py": 1738705056.0, "c:\\Users\\Jerson\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\abc.py": 1738705056.0, "c:\\Users\\Jerson\\AppData\\Local\\Programs\\Pyth

In [2]:
#| label: tbl-calidad-datos
#| tbl-colwidths:
#|   - 40
#|   - 20
#|   - 20
#|   - 20
#|   - 20
#|   - 20
#|   - 20
#|   - 20
#| tbl-cap: Resumen de Outliers por Variable
import pandas as pd
from IPython.display import display, Markdown
from tabulate import tabulate
import textwrap
import numpy as np
from scipy import stats
from sklearn.ensemble import IsolationForest

df = pd.read_csv("data/base10.csv")

# Función para envolver texto largo
def wrap_text(text, width):
    if isinstance(text, str) and len(text) > width:
        return '\n'.join(textwrap.wrap(text, width=width))
    return text

def detectar_outliers_iqr(df, column, factor=1.5):
    """Detección de outliers usando rango intercuartílico"""
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - factor * IQR
    upper_bound = Q3 + factor * IQR
    
    return (df[column] < lower_bound) | (df[column] > upper_bound)

def detectar_outliers_zscore(df, column, threshold=3):
    """Detección usando Z-score"""
    z_scores = np.abs(stats.zscore(df[column]))
    return z_scores > threshold

# Identificar variables numéricas (excluyendo la variable objetivo)
variables_numericas = df.select_dtypes(include=[np.number]).columns
if 'FLG_CLI_DEF60' in variables_numericas:
    variables_numericas = variables_numericas.drop('FLG_CLI_DEF60')
    
    # Análisis de outliers por variable
    outlier_summary = []
    
    for col in variables_numericas:
        try:
            outliers_iqr = detectar_outliers_iqr(df, col)
            outliers_zscore = detectar_outliers_zscore(df, col)
            
            outlier_summary.append({
                'Variable': col,
                'Outliers_IQR_Count': outliers_iqr.sum(),
                'Outliers_IQR_Pct': (outliers_iqr.sum() / len(df)) * 100,
                'Outliers_ZScore_Count': outliers_zscore.sum(),
                'Outliers_ZScore_Pct': (outliers_zscore.sum() / len(df)) * 100,
                'Max_Value': df[col].max(),
                'Q99': df[col].quantile(0.99),
                'Q95': df[col].quantile(0.95)
            })
        except Exception as e:
            display(Markdown(f"⚠️ **Error procesando variable {col}:** {str(e)}"))
    
    # Crear DataFrame y formatear
    outlier_df = pd.DataFrame(outlier_summary)
    outlier_df['Variable'] = outlier_df['Variable'].apply(lambda x: wrap_text(str(x), 7))
    
    for col in ['Outliers_IQR_Pct', 'Outliers_ZScore_Pct', 'Max_Value', 'Q99', 'Q95']:
        outlier_df[col] = outlier_df[col].map('{:.2f}'.format)
    
    # Renombrar columnas
    outlier_df.rename(columns={
        'Outliers_IQR_Count': 'Outliers IQR (#)',
        'Outliers_IQR_Pct': 'Outliers IQR (%)',
        'Outliers_ZScore_Count': 'Outliers Z-Score (#)',
        'Outliers_ZScore_Pct': 'Outliers Z-Score (%)',
        'Max_Value': 'Valor Máximo',
        'Q99': 'Percentil 99',
        'Q95': 'Percentil 95'
    }, inplace=True)
    
    # Mostrar tabla
    display(Markdown(tabulate(outlier_df, headers='keys', showindex=False)))

Variable      Outliers IQR (#)    Outliers IQR (%)    Outliers Z-Score (#)    Outliers Z-Score (%)    Valor Máximo    Percentil 99    Percentil 95
----------  ------------------  ------------------  ----------------------  ----------------------  --------------  --------------  --------------
MAX_ATR                   5497               10.87                     504                    1     1034                     51              29
_I_12M
NMES_AT                   5650               11.17                    1278                    2.53     6                      5               2
R15_I_U
6K_24M
RTOT_DA                   2068                4.09                     766                    1.51     1                      1               1
CT_DTOT
_24M
VAR_PRO                   4749                9.39                     817                    1.61     2.86                   0.92            0.5
M_ENT_A
CTU_24M
NMES_UA                   6194               12.24                    1706                    3.37    23                     21              14
TR3_I_2
4M
VAR_PRO                   2984                5.9                       15                    0.03  1754.15                   3.46            1.26
M_DEUDI
R_ACTU_
12M
INC_SUM                   3540                7                        116                    0.23     5.55556e+09          137.68           55.76
_ACTSH_
ACTU_24
M
MAX_ENT                     33                0.07                     244                    0.48     9                      5               4
_12M
N_NOR_2                     58                0.11                      28                    0.06    24                     24              24
4M
NMES_UA                   1984                3.92                    1264                    2.5     21                      3               0
TR15_I_
U6K_24M